# NB13 — subject-level mixed-effects analysis of the factorial (CPU)

Fits p_true ~ representation x graph + true class + (1|subject) + (1|split) + (1|cell x split) on the committed runs of the selection folds, the confirmation folds and the external cohort (ds004584).

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_mixed.py', log=f'{WORK}/pytest_mixed.txt')

In [ ]:
sh(f'{PY} mixed --runs {SRC}/reports/grid/runs_s0.jsonl {SRC}/reports/grid/runs_s1.jsonl {SRC}/reports/grid/runs_s2.jsonl --tag main --out {WORK}/mixed --prefix selection', log=f'{WORK}/mixed_selection.log')

In [ ]:
sh(f'{PY} mixed --runs {SRC}/reports/confirm/confirm_runs_s0.jsonl {SRC}/reports/confirm/confirm_runs_s1.jsonl --tag confirm --out {WORK}/mixed --prefix confirmation', log=f'{WORK}/mixed_confirmation.log')

In [ ]:
sh(f'{PY} mixed --runs {SRC}/reports/external/ext_runs_s0.jsonl {SRC}/reports/external/ext_runs_s1.jsonl --tag main --out {WORK}/mixed --prefix external', log=f'{WORK}/mixed_external.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)